In [22]:
# Import necessary libraries for deep learning, numerical operations, and plotting
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.optim import Adam

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [23]:
# Define problem domain boundaries
X_MIN, X_MAX = 0.0, 1.0
T_MIN, T_MAX = 0.0, 0.5

# Sensor configuration: how many sensors per time slice and at which times
N_SENSORS_PER_SLICE = 10
SENSOR_TIMES = [0.05, 0.20, 0.40]
N_SENSORS = N_SENSORS_PER_SLICE * len(SENSOR_TIMES)

# Generate sensor locations in space (x) and time (t)
SENSOR_X = np.tile(np.linspace(0.05, 0.95, N_SENSORS_PER_SLICE), len(SENSOR_TIMES))
SENSOR_T = np.repeat(SENSOR_TIMES, N_SENSORS_PER_SLICE)

# Number of collocation points for physics-informed training
N_COLLOC = 400

# Range for the unknown diffusion coefficient (xi)
XI_MIN, XI_MAX = 0.05, 0.5
N_TRAIN_SAMPLES = 800
N_VAL_SAMPLES   = 100

# DeepONet architecture parameters
P             = 200
BRANCH_HIDDEN = [256, 128, 64]
TRUNK_HIDDEN  = [256, 128, 64]

# Identifier network (MLP) parameters
IDENT_HIDDEN = [64, 64]

# Noise level added to sensor measurements
OBS_NOISE_STD = 0.05

# Hyperparameters for physics loss and prior
SIGMA_PDE  = 0.15
SIGMA_OBS  = OBS_NOISE_STD
MU0, SIGMA0 = 0.275, 0.15

# Weight for entropy regularization in the identifier loss
ENTROPY_WEIGHT = 0.3

# Training hyperparameters for DeepONet and Identifier
LR_DEEPONET = 5e-4
LR_IDENT    = 1e-4
EPOCHS_DEEPONET = 6000
EPOCHS_IDENT    = 5000
BATCH_SIZE = 64

# Hamiltonian Monte Carlo (HMC) sampling parameters
HMC_N_SAMPLES  = 3000
HMC_BURN_IN    = 500
HMC_STEP_SIZE  = 0.01
HMC_LEAPFROG_L = 40

# Set device (GPU if available, otherwise CPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


In [24]:
# Analytic solution to the 1D diffusion (heat) equation: u(x,t) = sin(pi*x) * exp(-xi * pi^2 * t)
def analytic_solution(x, t, xi):
    return np.sin(np.pi * x) * np.exp(-xi * np.pi**2 * t)


def analytic_solution_torch(x, t, xi):
    return torch.sin(np.pi * x) * torch.exp(-xi * (np.pi**2) * t)


# Generate synthetic training and validation datasets with noisy sensor measurements
def generate_dataset(n_samples, xi_min, xi_max, noise_std=OBS_NOISE_STD):
    xi_vals = np.random.uniform(xi_min, xi_max, n_samples)

    nx_q, nt_q = 20, 20
    xq = np.linspace(0.05, 0.95, nx_q)
    tq = np.linspace(0.02, T_MAX, nt_q)
    XX, TT = np.meshgrid(xq, tq)
    query_x = XX.ravel()
    query_t = TT.ravel()
    query_xt = np.stack([query_x, query_t], axis=1)
    N_QUERY = len(query_x)

    us_all = np.zeros((n_samples, N_SENSORS))
    u_full = np.zeros((n_samples, N_QUERY))

    for i, xi in enumerate(xi_vals):
        u_sensors = analytic_solution(SENSOR_X, SENSOR_T, xi)
        us_all[i] = u_sensors + noise_std * np.random.randn(N_SENSORS)
        u_full[i] = analytic_solution(query_x, query_t, xi)

    return xi_vals, us_all, u_full, query_xt


print("Generating training data...")
xi_train, us_train, uf_train, query_xt = generate_dataset(N_TRAIN_SAMPLES, XI_MIN, XI_MAX)
xi_val,   us_val,   uf_val,   _        = generate_dataset(N_VAL_SAMPLES,   XI_MIN, XI_MAX)

# Generate collocation points for enforcing the PDE (physics loss)
xc_near = np.random.uniform(0.05, 0.95, 200)
tc_near = np.random.uniform(0.05, 0.25, 200)
xc_far  = np.random.uniform(0.05, 0.95, 200)
tc_far  = np.random.uniform(0.02, T_MAX, 200)
xc = np.concatenate([xc_near, xc_far])
tc = np.concatenate([tc_near, tc_far])
colloc_xt = np.stack([xc, tc], axis=1)


def to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype).to(DEVICE)


# Convert all numpy arrays to PyTorch tensors on the selected device
us_train_t  = to_tensor(us_train)
uf_train_t  = to_tensor(uf_train)
us_val_t    = to_tensor(us_val)
uf_val_t    = to_tensor(uf_val)
query_xt_t  = to_tensor(query_xt)
colloc_xt_t = to_tensor(colloc_xt)
xi_train_t  = to_tensor(xi_train)

sensor_x_t = to_tensor(SENSOR_X)
sensor_t_t = to_tensor(SENSOR_T)

print(f"Train samples: {N_TRAIN_SAMPLES}, Val samples: {N_VAL_SAMPLES}")
print(f"Sensors: {N_SENSORS}, Query pts: {query_xt.shape[0]}, Colloc pts: {N_COLLOC}")

Generating training data...
Train samples: 800, Val samples: 100
Sensors: 30, Query pts: 400, Colloc pts: 400


In [25]:
# Helper function to build a simple multi-layer perceptron (MLP)
def make_mlp(in_dim, hidden_dims, out_dim, activation=nn.Tanh):
    layers = []
    prev = in_dim
    for h in hidden_dims:
        layers += [nn.Linear(prev, h), activation()]
        prev = h
    layers.append(nn.Linear(prev, out_dim))
    return nn.Sequential(*layers)


# DeepONet architecture: Branch net processes sensor data, Trunk net processes (x,t) locations
class DeepONet(nn.Module):
    def __init__(self, m, p, branch_hidden, trunk_hidden):
        super().__init__()
        self.branch = make_mlp(m, branch_hidden, p)
        self.trunk  = make_mlp(2, trunk_hidden, p)
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, us, xt):
        b = self.branch(us)
        v = self.trunk(xt)
        return b @ v.T + self.bias

    def predict_single(self, us, xt):
        b = self.branch(us.unsqueeze(0))
        v = self.trunk(xt)
        return (b @ v.T).squeeze(0) + self.bias.squeeze()


# Instantiate and prepare DeepONet for training
gtheta = DeepONet(N_SENSORS, P, BRANCH_HIDDEN, TRUNK_HIDDEN).to(DEVICE)
opt_g = Adam(gtheta.parameters(), lr=LR_DEEPONET)
best_val_g = float('inf')
gtheta_losses = []

print("\n--- Training DeepONet ---")

# Train the DeepONet to learn the solution operator
for epoch in range(1, EPOCHS_DEEPONET + 1):
    gtheta.train()
    idx = torch.randperm(N_TRAIN_SAMPLES)[:BATCH_SIZE]
    pred = gtheta(us_train_t[idx], query_xt_t)
    loss = ((pred - uf_train_t[idx])**2).mean()

    opt_g.zero_grad()
    loss.backward()
    opt_g.step()

    if epoch % 200 == 0:
        gtheta.eval()
        with torch.no_grad():
            val_loss = ((gtheta(us_val_t, query_xt_t) - uf_val_t)**2).mean().item()
        gtheta_losses.append((epoch, loss.item(), val_loss))
        print(f"  Epoch {epoch:4d} | train MSE {loss.item():.6f} | val MSE {val_loss:.6f}")
        if val_loss < best_val_g:
            best_val_g = val_loss
            torch.save(gtheta.state_dict(), '/tmp/gtheta_best.pt')

# Load the best DeepONet model
gtheta.load_state_dict(torch.load('/tmp/gtheta_best.pt', map_location=DEVICE))
gtheta.eval()
print("DeepONet training complete.")


--- Training DeepONet ---
  Epoch  200 | train MSE 0.003560 | val MSE 0.003808
  Epoch  400 | train MSE 0.001893 | val MSE 0.001932
  Epoch  600 | train MSE 0.002016 | val MSE 0.001486
  Epoch  800 | train MSE 0.001529 | val MSE 0.001762
  Epoch 1000 | train MSE 0.001044 | val MSE 0.001024
  Epoch 1200 | train MSE 0.000613 | val MSE 0.000571
  Epoch 1400 | train MSE 0.000449 | val MSE 0.000510
  Epoch 1600 | train MSE 0.000547 | val MSE 0.000385
  Epoch 1800 | train MSE 0.000273 | val MSE 0.000290
  Epoch 2000 | train MSE 0.000273 | val MSE 0.000322
  Epoch 2200 | train MSE 0.000545 | val MSE 0.000258
  Epoch 2400 | train MSE 0.000252 | val MSE 0.000233
  Epoch 2600 | train MSE 0.000229 | val MSE 0.000330
  Epoch 2800 | train MSE 0.000193 | val MSE 0.000257
  Epoch 3000 | train MSE 0.000380 | val MSE 0.000260
  Epoch 3200 | train MSE 0.000475 | val MSE 0.000620
  Epoch 3400 | train MSE 0.000270 | val MSE 0.000301
  Epoch 3600 | train MSE 0.000563 | val MSE 0.000400
  Epoch 3800 | trai

In [26]:
print("\nPrecomputing AD derivatives at collocation points...")


# Compute time derivative (ut) and second spatial derivative (uxx) using automatic differentiation
def compute_derivatives_for_us(us_vec, colloc_xt):
    ut_list  = []
    uxx_list = []
    b_vec = gtheta.branch(us_vec.unsqueeze(0)).squeeze(0).detach()

    for k in range(colloc_xt.shape[0]):
        xt_k = colloc_xt[k].clone().detach().requires_grad_(True)
        v_vec = gtheta.trunk(xt_k.unsqueeze(0)).squeeze(0)
        u_k   = (b_vec * v_vec).sum() + gtheta.bias.squeeze()

        grad1 = torch.autograd.grad(u_k, xt_k, create_graph=True)[0]
        ut_k  = grad1[1]

        grad2 = torch.autograd.grad(grad1[0], xt_k, create_graph=False)[0]
        uxx_k = grad2[0]

        ut_list.append(ut_k.detach())
        uxx_list.append(uxx_k.detach())

    return torch.stack(ut_list), torch.stack(uxx_list)


# Precompute derivatives for all training samples to speed up later training
ut_all  = torch.zeros(N_TRAIN_SAMPLES, N_COLLOC, device=DEVICE)
uxx_all = torch.zeros(N_TRAIN_SAMPLES, N_COLLOC, device=DEVICE)

for i in range(N_TRAIN_SAMPLES):
    if (i + 1) % 100 == 0:
        print(f"  Derivatives: {i+1}/{N_TRAIN_SAMPLES}")
    ut_all[i], uxx_all[i] = compute_derivatives_for_us(us_train_t[i], colloc_xt_t)

print(f"Derivative precomputation complete. Shape: ut_all = {ut_all.shape}")

# Quick validation of derivative accuracy
with torch.no_grad():
    xc_t = torch.tensor(xc, dtype=torch.float32, device=DEVICE)
    tc_t = torch.tensor(tc, dtype=torch.float32, device=DEVICE)

    xi_check = 0.18
    ut_true  = -xi_check * np.pi**2 * np.sin(np.pi * xc) * np.exp(-xi_check * np.pi**2 * tc)
    uxx_true = -np.pi**2 * np.sin(np.pi * xc) * np.exp(-xi_check * np.pi**2 * tc)

    ut_pred  = ut_all[0].cpu().numpy()
    uxx_pred = uxx_all[0].cpu().numpy()

print(f"  ut  relative error: {np.abs(ut_pred - ut_true).mean() / np.abs(ut_true).mean():.4f}")
print(f"  uxx relative error: {np.abs(uxx_pred - uxx_true).mean() / np.abs(uxx_true).mean():.4f}")


Precomputing AD derivatives at collocation points...
  Derivatives: 100/800
  Derivatives: 200/800
  Derivatives: 300/800
  Derivatives: 400/800
  Derivatives: 500/800
  Derivatives: 600/800
  Derivatives: 700/800
  Derivatives: 800/800
Derivative precomputation complete. Shape: ut_all = torch.Size([800, 400])
  ut  relative error: 0.2342
  uxx relative error: 0.1459


In [27]:
# Neural network that identifies the unknown parameter xi from sensor measurements
class IdentifierMLP(nn.Module):
    def __init__(self, m, hidden_dims):
        super().__init__()
        self.net = make_mlp(m, hidden_dims, 2, activation=nn.Tanh)

    def forward(self, us):
        out   = self.net(us)
        mu    = out[:, 0]
        sigma = torch.nn.functional.softplus(out[:, 1]) + 1e-4
        return mu, sigma


# Evidence Lower Bound (ELBO) loss for training the identifier network
def elbo_loss(phi_net, us_b, ut_b, uxx_b):
    mu, sigma = phi_net(us_b)

    eps = torch.randn_like(mu)
    xi  = mu + sigma * eps

    xi_col    = xi.unsqueeze(1)
    residuals = ut_b - xi_col * uxx_b
    log_ppde  = -0.5 / (SIGMA_PDE ** 2) * (residuals ** 2).mean(dim=1)

    mu0_t  = torch.tensor(MU0,   device=DEVICE)
    sig0_t = torch.tensor(SIGMA0, device=DEVICE)
    kl = (torch.log(sig0_t / sigma) +
          (sigma**2 + (mu - mu0_t)**2) / (2.0 * sig0_t**2) - 0.5)

    entropy_reg = -ENTROPY_WEIGHT * torch.log(sigma + 1e-8).mean()

    SIGMA_PHI_FLOOR = 0.01
    sigma_floor_penalty = torch.relu(SIGMA_PHI_FLOOR - sigma).pow(2).mean()

    elbo = log_ppde - kl
    return -elbo.mean()


# Initialize and train the identifier network
iphi = IdentifierMLP(N_SENSORS, IDENT_HIDDEN).to(DEVICE)
opt_i = Adam(iphi.parameters(), lr=LR_IDENT)

print("\n--- Training Identifier MLP ---")
ident_losses = []

for epoch in range(1, EPOCHS_IDENT + 1):
    iphi.train()
    idx = torch.randperm(N_TRAIN_SAMPLES)[:BATCH_SIZE]

    loss = elbo_loss(
        iphi,
        us_b  = us_train_t[idx],
        ut_b  = ut_all[idx],
        uxx_b = uxx_all[idx],
    )

    opt_i.zero_grad()
    loss.backward()
    opt_i.step()

    if epoch % 300 == 0:
        iphi.eval()
        with torch.no_grad():
            mu_v, sig_v = iphi(us_val_t)
            mae = (mu_v - to_tensor(xi_val)).abs().mean().item()
        ident_losses.append((epoch, loss.item(), mae))
        print(f"  Epoch {epoch:4d} | -ELBO {loss.item():.4f} | val MAE {mae:.4f}")
        iphi.train()

iphi.eval()
print("Identifier MLP training complete.")


--- Training Identifier MLP ---
  Epoch  300 | -ELBO 15.3566 | val MAE 0.0914
  Epoch  600 | -ELBO 5.7110 | val MAE 0.0497
  Epoch  900 | -ELBO 4.2737 | val MAE 0.0317
  Epoch 1200 | -ELBO 3.8272 | val MAE 0.0296
  Epoch 1500 | -ELBO 3.4002 | val MAE 0.0278
  Epoch 1800 | -ELBO 3.7037 | val MAE 0.0278
  Epoch 2100 | -ELBO 3.6493 | val MAE 0.0275
  Epoch 2400 | -ELBO 3.5383 | val MAE 0.0265
  Epoch 2700 | -ELBO 3.4718 | val MAE 0.0282
  Epoch 3000 | -ELBO 3.3821 | val MAE 0.0259
  Epoch 3300 | -ELBO 3.5578 | val MAE 0.0247
  Epoch 3600 | -ELBO 3.4336 | val MAE 0.0283
  Epoch 3900 | -ELBO 3.7421 | val MAE 0.0253
  Epoch 4200 | -ELBO 3.6427 | val MAE 0.0255
  Epoch 4500 | -ELBO 3.4859 | val MAE 0.0246
  Epoch 4800 | -ELBO 3.4644 | val MAE 0.0257
Identifier MLP training complete.


In [28]:
# Compute potential energy and its gradient for HMC sampling
def compute_U_and_grad(xi_val_scalar, mu_star, sigma_star_hmc,
                       ut_k, uxx_k):
    if not np.isfinite(xi_val_scalar) or xi_val_scalar <= 0:
        return 1e10, np.sign(mu_star - xi_val_scalar) * 1e6

    xi_t = torch.tensor([xi_val_scalar], dtype=torch.float32,
                         device=DEVICE, requires_grad=True)

    residuals = ut_k - xi_t * uxx_k
    pde_term  = (residuals**2).mean() / (2.0 * SIGMA_PDE**2)

    prior_term = (xi_t - mu_star)**2 / (2.0 * sigma_star_hmc**2)

    U = pde_term + prior_term
    U.backward()
    dU = xi_t.grad.item()
    if not np.isfinite(dU):
        dU = np.sign(xi_val_scalar - mu_star) * 1e6
    return U.item(), dU


# Hamiltonian Monte Carlo sampler to draw samples from the posterior of xi
def hmc_sample(mu_star, sigma_star_hmc, ut_k, uxx_k,
               n_samples=HMC_N_SAMPLES, burn_in=HMC_BURN_IN,
               step_size=HMC_STEP_SIZE, L=HMC_LEAPFROG_L):
    xi_curr = float(mu_star)
    samples  = []
    n_accept = 0

    current_step  = step_size
    target_accept = 0.6
    adapt_rate    = 0.01
    adapt_steps   = burn_in

    def U_dU(xi):
        return compute_U_and_grad(xi, mu_star, sigma_star_hmc, ut_k, uxx_k)

    for i in range(n_samples + burn_in):
        p = np.random.randn()
        U_curr, _ = U_dU(xi_curr)
        H_curr = U_curr + 0.5 * p**2

        xi_prop = xi_curr
        p_prop  = p

        _, dU = U_dU(xi_prop)
        p_prop = p_prop - 0.5 * current_step * dU

        for _ in range(L):
            xi_prop = xi_prop + current_step * p_prop
            _, dU   = U_dU(xi_prop)
            p_prop  = p_prop - current_step * dU

        p_prop = p_prop + 0.5 * current_step * dU

        U_prop, _ = U_dU(xi_prop)
        H_prop = U_prop + 0.5 * p_prop**2

        log_alpha = -(H_prop - H_curr)
        if np.log(np.random.rand() + 1e-300) < log_alpha:
            xi_curr  = xi_prop
            n_accept += 1

        if i < adapt_steps:
            rate = n_accept / (i + 1)
            if rate < target_accept:
                current_step *= (1 - adapt_rate)
            else:
                current_step *= (1 + adapt_rate)
            current_step = np.clip(current_step, 1e-4, 0.1)

        if i >= burn_in:
            samples.append(xi_curr)

    acceptance_rate = n_accept / (n_samples + burn_in)
    return np.array(samples), acceptance_rate

In [32]:
print("\n--- Running Inference on Test Observation ---")

# True (unknown) diffusion coefficient for the test case
XI_STAR = 0.18

# Generate noisy sensor data for the test observation
us_test_np  = analytic_solution(SENSOR_X, SENSOR_T, XI_STAR)
us_test_np += OBS_NOISE_STD * np.random.randn(N_SENSORS)
us_test_t   = to_tensor(us_test_np).unsqueeze(0)

# Get prior mean and variance from the trained identifier network
with torch.no_grad():
    mu_phi, sigma_phi = iphi(us_test_t)
    mu_star    = mu_phi.item()
    sigma_star = sigma_phi.item()

print(f"  Identifier prior mean  = {mu_star:.4f}")
print(f"  Identifier prior std   = {sigma_star:.4f}")

print("  Computing AD derivatives for test observation...")
ut_test, uxx_test = compute_derivatives_for_us(us_test_t.squeeze(0), colloc_xt_t)

print("\n  --- PDE least-squares check ---")
with torch.no_grad():
    num   = (ut_test * uxx_test).sum()
    den   = (uxx_test**2).sum() + 1e-12
    xi_ls = (num / den).item()
    print(f"  PDE-only xi_ls = {xi_ls:.4f}  (true xi* = {XI_STAR:.4f})")

# Least-squares estimate using only sensor data
xi_vals_scan = np.linspace(0.15, 0.22, 200)
resids = [((us_test_np - analytic_solution(SENSOR_X, SENSOR_T, xi))**2).sum()
          for xi in xi_vals_scan]
xi_obs_ls = xi_vals_scan[np.argmin(resids)]
print(f"  Sensor-only xi_ls = {xi_obs_ls:.4f}  (true xi* = {XI_STAR:.4f})")

print(f"\n  Running HMC ({HMC_N_SAMPLES} samples, burn-in {HMC_BURN_IN})...")
sigma_star_hmc = max(sigma_star, 0.05)
posterior_samples, acc_rate = hmc_sample(
    mu_star, sigma_star_hmc, ut_test, uxx_test)

xi_mean = posterior_samples.mean()
xi_std  = posterior_samples.std()
ci_low  = np.percentile(posterior_samples, 2.5)
ci_high = np.percentile(posterior_samples, 97.5)

print(f"\n  HMC acceptance rate   = {acc_rate:.2%}")
print(f"  Posterior mean        = {xi_mean:.4f}  (true: {XI_STAR:.4f})")
print(f"  Posterior std         = {xi_std:.4f}")
print(f"  95% credible interval = [{ci_low:.4f}, {ci_high:.4f}]")
print(f"  True xi* in CI?       = {ci_low <= XI_STAR <= ci_high}")


--- Running Inference on Test Observation ---
  Identifier prior mean  = 0.1822
  Identifier prior std   = 0.0259
  Computing AD derivatives for test observation...

  --- PDE least-squares check ---
  PDE-only xi_ls = 0.1810  (true xi* = 0.1800)
  Sensor-only xi_ls = 0.1803  (true xi* = 0.1800)

  Running HMC (3000 samples, burn-in 500)...

  HMC acceptance rate   = 92.06%
  Posterior mean        = 0.1811  (true: 0.1800)
  Posterior std         = 0.0248
  95% credible interval = [0.1322, 0.2270]
  True xi* in CI?       = True


In [33]:
# Generate visualization of results: DeepONet predictions, training curves, and HMC posterior
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("DeepONet + Identifier MLP + HMC - 1D Diffusion Equation", fontsize=13)

ax = axes[0, 0]
x_plot = np.linspace(0.01, 0.99, 100)
t_plot = 0.2
u_true = analytic_solution(x_plot, t_plot, XI_STAR)
xt_plot = to_tensor(np.stack([x_plot, np.full_like(x_plot, t_plot)], axis=1))
with torch.no_grad():
    u_pred = gtheta.predict_single(us_test_t.squeeze(0), xt_plot).cpu().numpy()
ax.plot(x_plot, u_true, 'k-',  lw=2,   label=f'True (xi={XI_STAR})')
ax.plot(x_plot, u_pred, 'b--', lw=1.5, label='DeepONet prediction')
ax.scatter(SENSOR_X, us_test_np, c='r', s=20, zorder=5, label='Sensors (noisy)')
ax.set_title(f'DeepONet prediction at t={t_plot}')
ax.set_xlabel('x'); ax.set_ylabel('u(x,t)')
ax.legend(fontsize=8)

ax = axes[0, 1]
if gtheta_losses:
    ep, tr, vl = zip(*gtheta_losses)
    ax.semilogy(ep, tr, label='Train MSE')
    ax.semilogy(ep, vl, label='Val MSE')
ax.set_title('DeepONet training loss')
ax.set_xlabel('Epoch'); ax.legend(fontsize=8)

ax = axes[0, 2]
with torch.no_grad():
    mu_val, sig_val = iphi(us_val_t)
    mu_val  = mu_val.cpu().numpy()
    sig_val = sig_val.cpu().numpy()
ax.scatter(xi_val, mu_val, s=6, alpha=0.5)
xlim = [XI_MIN, XI_MAX]
ax.plot(xlim, xlim, 'r--', lw=1, label='perfect')
ax.set_title('Identifier: predicted mean vs true xi (val)')
ax.set_xlabel('True xi'); ax.set_ylabel('Predicted mean')
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.hist(posterior_samples, bins=60, density=True, alpha=0.7,
        color='steelblue', label='HMC posterior')
ax.axvline(XI_STAR,  color='k',      lw=2, linestyle='-',
           label=f'True xi* = {XI_STAR}')
ax.axvline(xi_mean,  color='red',    lw=2, linestyle='--',
           label=f'Post. mean = {xi_mean:.3f}')
ax.axvline(mu_star,  color='orange', lw=2, linestyle=':',
           label=f'Prior mean = {mu_star:.3f}')
ax.axvspan(ci_low, ci_high, alpha=0.15, color='steelblue', label='95% CI')
ax.set_title('Posterior P(xi | data)')
ax.set_xlabel('xi'); ax.set_ylabel('Density')
ax.legend(fontsize=7)

ax = axes[1, 1]
ax.plot(np.arange(len(posterior_samples)), posterior_samples,
        lw=0.5, color='steelblue', alpha=0.8)
ax.axhline(XI_STAR, color='k', lw=1.5, linestyle='--',
           label=f'True xi* = {XI_STAR}')
ax.axhline(xi_mean, color='r', lw=1,   linestyle='--',
           label=f'Post. mean = {xi_mean:.3f}')
ax.set_title('HMC trace (post burn-in)')
ax.set_xlabel('Sample index'); ax.set_ylabel('xi')
ax.legend(fontsize=8)

ax = axes[1, 2]
if ident_losses:
    ep2, neg_elbo, mae2 = zip(*ident_losses)
    ax2 = ax.twinx()
    ax.plot(ep2, neg_elbo, 'b-',  label='-ELBO (left)')
    ax2.plot(ep2, mae2,    'r--', label='MAE (right)')
    ax.set_ylabel('-ELBO', color='b')
    ax2.set_ylabel('MAE', color='r')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7)
ax.set_title('Identifier MLP training')
ax.set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('/tmp/deeponet_hmc_results_fixed.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved.")



Figure saved.


In [34]:
# Print final summary of results
print("\n" + "="*55)
print("SUMMARY")
print("="*55)
print(f"  True xi*                 : {XI_STAR:.4f}")
print(f"  Identifier prior mean    : {mu_star:.4f}")
print(f"  Posterior mean           : {xi_mean:.4f}")
print(f"  Posterior std            : {xi_std:.4f}")
print(f"  95% credible interval    : [{ci_low:.4f}, {ci_high:.4f}]")
print(f"  True xi* in CI           : {ci_low <= XI_STAR <= ci_high}")
print(f"  HMC acceptance rate      : {acc_rate:.2%}")
print("="*55)


SUMMARY
  True xi*                 : 0.1800
  Identifier prior mean    : 0.1822
  Posterior mean           : 0.1811
  Posterior std            : 0.0248
  95% credible interval    : [0.1322, 0.2270]
  True xi* in CI           : True
  HMC acceptance rate      : 92.06%
